# 🛡️ PhaseGuard - Layer 1 Model Training & Pipeline Notebook
This notebook provides an interactive workspace inside Jupyter to compile your custom voice recordings, train the Layer 1 deepfake classifier, and enroll user voiceprints.

---

### ⚠️ Environment Verification
Before running the cells, make sure you have activated your conda environment and launched Jupyter:
1. Open **Anaconda Prompt**
2. Run: `conda activate phaseguard`
3. Run: `jupyter notebook`
4. Open this notebook and ensure **`Python (PhaseGuard)`** is selected in the top-right kernel selector.

In [4]:
# Step 1: Mock lazy modules to prevent Windows import checks from failing
import sys
from unittest.mock import MagicMock
sys.modules['k2'] = MagicMock()
sys.modules['flair'] = MagicMock()
sys.modules['flair.data'] = MagicMock()
sys.modules['flair.embeddings'] = MagicMock()
sys.modules['flair.models'] = MagicMock()
sys.modules['spacy'] = MagicMock()
sys.modules['spacy.tokens'] = MagicMock()

print("Mocks applied successfully!")

Mocks applied successfully!


### Step 2: Import Custom Modules
Let's import our preprocessing code and neural network loaders to verify paths.

In [10]:
import torch
import numpy as np
import librosa
import soundfile as sf

from preprocess import load_and_standardize, extract_5_signals, audio_to_mel_spectrogram
from train_layer1 import PhaseGuardL1

print("All modules imported successfully!")
print("PyTorch CUDA GPU available:", torch.cuda.is_available())

All modules imported successfully!
PyTorch CUDA GPU available: False


### Step 3: Compile Your Custom Recordings
Once you replace the synthetic WAV files inside `data/real_voices/[name]` and `data/clones/[name]` with your team's real voice recordings and ElevenLabs clones, run the compiler code cell below to rebuild the NumPy binary matrices.

In [11]:
from build_dataset import build_training_dataset

real_folders = [
    "data/real_voices/vishal",
    "data/real_voices/abhinav",
    "data/real_voices/aditya",
    "data/real_voices/dhruv"
]

fake_folders = [
    "data/clones/vishal",
    "data/clones/abhinav",
    "data/clones/aditya",
    "data/clones/dhruv"
]

print("Starting dataset compilation...")
build_training_dataset(real_folders, fake_folders, "data/processed")

Starting dataset compilation...
Processing REAL/HUMAN directories recursively...
Scanning: data/real_voices/vishal
  [OK] vishal_001.wav (Label: 0)
  [OK] vishal_002.wav (Label: 0)
  [OK] vishal_003.wav (Label: 0)
  [OK] vishal_004.wav (Label: 0)
  [OK] vishal_005.wav (Label: 0)
  [OK] vishal_006.wav (Label: 0)
  [OK] vishal_007.wav (Label: 0)
  [OK] vishal_008.wav (Label: 0)
  [OK] vishal_009.wav (Label: 0)
  [OK] vishal_010.wav (Label: 0)
  [OK] vishal_011.wav (Label: 0)
  [OK] vishal_012.wav (Label: 0)
  [OK] vishal_013.wav (Label: 0)
  [OK] vishal_014.wav (Label: 0)
  [OK] vishal_015.wav (Label: 0)
  [OK] vishal_016.wav (Label: 0)
  [OK] vishal_017.wav (Label: 0)
  [OK] vishal_018.wav (Label: 0)
  [OK] vishal_019.wav (Label: 0)
  [OK] vishal_020.wav (Label: 0)
  [OK] vishal_021.wav (Label: 0)
  [OK] vishal_022.wav (Label: 0)
  [OK] vishal_023.wav (Label: 0)
  [OK] vishal_024.wav (Label: 0)
  [OK] vishal_025.wav (Label: 0)
  [OK] vishal_026.wav (Label: 0)
  [OK] vishal_027.wav (Labe

### Step 3b: Alternative - Multi-Source, Speaker-Aware Split
If you want to train on multiple datasets (such as WaveFake, Common Voice Hindi, ASVspoof, etc.) and perform a speaker-aware split to avoid train/test data leakage, run `process_phase1.py` instead of the simple compiler above.

In [ ]:
%run process_phase1.py

### Step 4: Train Layer 1 CNN Model
Now we fine-tune the pre-trained MobileNetV3 CNN model to classify your actual voices vs. clones using the compiled dataset. If you ran Step 3b, the training script will automatically detect and load your speaker-aware splits.

In [12]:
from train_layer1 import train

print("Starting MobileNetV3 Training loop...")
train()

Starting MobileNetV3 Training loop...
Training using device: cpu
Loaded 400 samples.
Real (0): 200, Fake (1): 200

Starting Training Layer 1...
Epoch 02/20 | Train Loss: 0.0025 | Train Acc: 100.0% | Test Loss: 2.8316 | Test Acc: 43.8%
Epoch 04/20 | Train Loss: 0.0006 | Train Acc: 100.0% | Test Loss: 1.9672 | Test Acc: 43.8%
Epoch 06/20 | Train Loss: 0.0001 | Train Acc: 100.0% | Test Loss: 2.1089 | Test Acc: 43.8%
Epoch 08/20 | Train Loss: 0.0004 | Train Acc: 100.0% | Test Loss: 1.9709 | Test Acc: 43.8%
Epoch 10/20 | Train Loss: 0.0053 | Train Acc: 100.0% | Test Loss: 0.5179 | Test Acc: 71.2%
Epoch 12/20 | Train Loss: 0.0020 | Train Acc: 100.0% | Test Loss: 0.7364 | Test Acc: 43.8%
Epoch 14/20 | Train Loss: 0.0001 | Train Acc: 100.0% | Test Loss: 1.3329 | Test Acc: 43.8%
Epoch 16/20 | Train Loss: 0.0001 | Train Acc: 100.0% | Test Loss: 1.3617 | Test Acc: 43.8%
Epoch 18/20 | Train Loss: 0.0001 | Train Acc: 100.0% | Test Loss: 1.1533 | Test Acc: 41.2%
Epoch 20/20 | Train Loss: 0.0000 | Tr

### Step 5: Enroll Speaker Profiles (Layer 2)
After Layer 1 is trained, extract and average your team's real voiceprints using the SpeechBrain ECAPA model to build the authentic voice credentials database.

In [ ]:
import enroll_voices

print("Running Speaker Profile Enrollment...")
enroll_voices.main()

Running Speaker Profile Enrollment...
Enrolling speaker: vishal
  Processed: vishal_001.wav
  Processed: vishal_002.wav
  Processed: vishal_003.wav
  Processed: vishal_004.wav
  Processed: vishal_005.wav
  Processed: vishal_006.wav
  Processed: vishal_007.wav
  Processed: vishal_008.wav
  Processed: vishal_009.wav
  Processed: vishal_010.wav
  Processed: vishal_011.wav
  Processed: vishal_012.wav
  Processed: vishal_013.wav
  Processed: vishal_014.wav
  Processed: vishal_015.wav
  Processed: vishal_016.wav
  Processed: vishal_017.wav
  Processed: vishal_018.wav
  Processed: vishal_019.wav
  Processed: vishal_020.wav
  Processed: vishal_021.wav
  Processed: vishal_022.wav
  Processed: vishal_023.wav
  Processed: vishal_024.wav
  Processed: vishal_025.wav
  Processed: vishal_026.wav
  Processed: vishal_027.wav
  Processed: vishal_028.wav
  Processed: vishal_029.wav
  Processed: vishal_030.wav
  Processed: vishal_031.wav
  Processed: vishal_032.wav
  Processed: vishal_033.wav
  Processed:

### Step 6: Verify Database Existence
Run this cell to double-check that your model weights and profiles exist and are ready for the Streamlit dashboard.

In [ ]:
import os

print("Layer 1 Weights (layer1_mobilenet.pth) exists:", os.path.exists("models/layer1_mobilenet.pth"))
print("Layer 2 Profiles (voiceprints.pth) exists:", os.path.exists("models/voiceprints.pth"))